In [0]:
%sql
select * from `public-data`.cpdoc.dhbb_bronze where content like '%Marta Suplicy%'

In [0]:
# Instalação de dependências
%pip install databricks-vectorsearch
%pip install sentence-transformers
%pip install langchain
%pip install mlflow

# Imports
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, concat_ws, lit, row_number
from pyspark.sql.window import Window
import os

In [0]:
# PARTE 3: Chunking (Dividir em Pedaços)
# Documentos grandes precisam ser divididos em chunks menores para embedding eficiente.
# Passo 3.1: Função de Chunking

from pyspark.sql.functions import udf, explode
from pyspark.sql.types import ArrayType, StringType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def chunk_text(text, chunk_size=500, overlap=100):
    """
    Divide texto em chunks com sobreposição
    
    Args:
        text: Texto a ser dividido
        chunk_size: Tamanho de cada chunk (caracteres)
        overlap: Sobreposição entre chunks
    
    Returns:
        Lista de chunks
    """

    if not text or len(text.strip()) == 0:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if len(chunk) > 20:
            chunks.append(chunk)
        start = end - overlap
    return chunks if chunks else []

chunk_udf = udf(chunk_text, ArrayType(StringType()))

columns=[
        "id",
        "file_name",
        "content",
        "person_name",
        "natureza",
        "sexo",
        "cargos"
    ]

df_chunked = spark.table('`public-data`.cpdoc.dhbb_bronze') \
    .select('file_name', chunk_udf('content').alias('chunks')) \
    .select('file_name', explode('chunks').alias('chunk_text'))

window = Window.orderBy('file_name')
df_chunked = df_chunked.withColumn(
    'chunk_id',
    row_number().over(window)
)

df_chunked.write.format('delta').mode('overwrite').saveAsTable(
    '`public-data`.cpdoc.dhbb_prata_chunks'
)

# print(f"✓ {df_chunked.count()} chunks criados!")
display(df_chunked)

In [0]:
# PARTE 4: Gerar Embeddings - 3h46' rodando
# Passo 4.1: Usando Transformers do Hugging Face

from sentence_transformers import SentenceTransformer
import numpy as np

# Baixar e carregar modelo de embedding (small para Free tier)
#databricks-gte-large-en
model = SentenceTransformer('all-MiniLM-L6-v2')  # Pequeno e rápido

print(f"Dimensão de embeddings: {model.get_sentence_embedding_dimension()}")

# Função para gerar embedding
def generate_embedding(text):
    """Gera embedding para um texto"""
    if not text:
        return None
    embedding = model.encode(text)
    return embedding.tolist()  # Converter para lista para Spark

# Registrar como UDF
from pyspark.sql.types import ArrayType, DoubleType

embedding_udf = udf(generate_embedding, ArrayType(DoubleType()))

# Aplicar embeddings aos chunks
df_embeddings = spark.table('`public-data`.cpdoc.dhbb_prata_chunks') \
    .withColumn('embedding', embedding_udf('chunk_text'))

# Salvar com embeddings
df_embeddings.write.format('delta').mode('overwrite').saveAsTable(
    '`public-data`.cpdoc.dhbb_prata_with_embeddings'
)

print("✓ Embeddings gerados e salvos!")
df_embeddings.display()

In [0]:
### Passo 4.2: Habilitar Vector Search (Importante!)

# Habilitar Vector Search na tabela
spark.sql("""
    ALTER TABLE `public-data`.cpdoc.dhbb_prata_with_embeddings
    SET TBLPROPERTIES ('delta.enableRowTracking' = 'true')
""")

print("✓ Vector Search habilitado na tabela!")

In [0]:
## PARTE 5: Criar Endpoint de Vector Search
### Passo 5.1: Criar Vector Search Endpoint

from databricks.vector_search.client import VectorSearchClient

# Inicializar cliente
client = VectorSearchClient()

# Parâmetros do endpoint
endpoint_name = "dhbb-rag-endpoint"
index_name = "dhbb-documents-index"

# Criar ou recuperar endpoint
try:
    endpoints = client.list_endpoints()
    endpoint_exists = any(ep.get('name') == endpoint_name for ep in endpoints.get('endpoints', []))
    
    if not endpoint_exists:
        print(f"Criando endpoint: {endpoint_name}...")
        client.create_endpoint(
            name=endpoint_name,
            endpoint_type="STANDARD"
        )
        print(f"✓ Endpoint criado: {endpoint_name}")
    else:
        print(f"✓ Endpoint já existe: {endpoint_name}")
except Exception as e:
    print(f"Nota: {e}")

In [0]:
### Passo 5.2: Criar Index no Vector Search

import time

try:
    # Corrigir index_name para formato <catalog>.<schema>.<table> e usar apenas alfanuméricos/underscores
    index_name = "public_data_cpdoc_dhbb_prata_with_embeddings"
    full_index_name = "public-data.cpdoc.dhbb_prata_with_embeddings"

    index = client.create_delta_sync_index(
        endpoint_name=endpoint_name,
        index_name=full_index_name,
        primary_key="chunk_id",
        embedding_dimension=384,  # Dimensão do all-MiniLM-L6-v2
        embedding_vector_column="embedding",
        source_table_name="`public-data`.cpdoc.dhbb_prata_with_embeddings",
        pipeline_type="BATCH"  # Alterado para BATCH conforme limitação do workspace
        # NÃO FUNCIONA COM DATABRICKS FREE
    )
    
    print(f"✓ Índice criado: {full_index_name}")
    
    time.sleep(10)
    
except Exception as e:
    print(f"Índice pode já existir: {e}")

print("✓ Vector Search configurado! Ver no painel e aguardar sair do status = Provisioning")

In [0]:
## PARTE 6: Implementar Busca Vetorial

### Passo 6.1: Função de Retrieval (Busca)

def retrieve_context(query_text, top_k=3):
    """
    Busca nos documentos os chunks mais relevantes
    
    Args:
        query_text: Pergunta do usuário
        top_k: Número de resultados a retornar
    
    Returns:
        Lista de chunks relevantes
    """
    from sentence_transformers import SentenceTransformer
    
    # Gerar embedding da pergunta
    model = SentenceTransformer('all-MiniLM-L6-v2')
    query_embedding = model.encode(query_text).tolist()
    
    # Buscar no Vector Search
    results = client.query_index(
        endpoint_name=endpoint_name,
        index_name=index_name,
        query_vector=query_embedding,
        num_results=top_k
    )
    
    # Extrair chunks do resultado
    retrieved_chunks = []
    for result in results.get('result', {}).get('data_array', []):
        if len(result) > 0:
            retrieved_chunks.append(result[0])
    
    return retrieved_chunks

# Testar busca
test_query = "Qual é o tema principal dos documentos?"
print(f"Buscando por: {test_query}")

context = retrieve_context(test_query, top_k=3)
print(f"Resultados encontrados: {len(context)}")
for i, chunk in enumerate(context):
    print(f"\n[{i+1}] {chunk[:200]}...")
